# Clean-reference regional specialist pilot — private research
New regularized heads, frozen external DINOv2-small, five outer/three inner PatientID-grouped folds.
**Not exact V13. Not confirmed patient independence. Historical 58-case gold is exploratory.**
Attach the competition and RSNA Knee V14 Patient Audit output (346132000), enable a GPU and internet for the pinned public encoder. No leaderboard submission.


In [ ]:
import sys, types, os
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
diagnostics = types.ModuleType('diagnostics')
sys.modules['diagnostics'] = diagnostics
exec('"""Keyed prediction diagnostics and paired, optionally grouped AUC confirmation.\n\nNo label fitting occurs here. Confidence intervals are not selection-adjusted;\nevaluate one frozen candidate on a confirmation cohort not used for selection.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import rankdata\n\nUID = \'StudyInstanceUID\'\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\n\n\ndef align(frame, ids=None, labels=False):\n    if frame.empty or not set([UID] + TARGETS).issubset(frame):\n        raise ValueError(\'Empty table or missing study/target columns\')\n    if frame[UID].isna().any() or frame[UID].duplicated().any():\n        raise ValueError(\'Missing or duplicate study IDs\')\n    if not frame[UID].map(lambda x: isinstance(x, str) and bool(x.strip())).all():\n        raise ValueError(\'Study IDs must be nonempty strings; read CSV with dtype=str for IDs\')\n    if ids is not None:\n        if len(ids) != len(set(ids)) or set(ids) != set(frame[UID]):\n            raise ValueError(\'Study coverage mismatch; partial scoring is prohibited\')\n        frame = frame.set_index(UID).loc[list(ids)].reset_index()\n    values = frame[TARGETS].to_numpy(float)\n    good = np.isnan(values) | (values == 0) | (values == 1) if labels else (\n        np.isfinite(values) & (values >= 0) & (values <= 1))\n    if not good.all():\n        raise ValueError(\'Labels must be binary or missing\' if labels else \'Invalid probabilities/ranks\')\n    return frame[[UID] + TARGETS].copy()\n\n\ndef auc(y, p):\n    valid = np.isfinite(y)\n    y, p = y[valid], p[valid]\n    pos, neg = np.sum(y == 1), np.sum(y == 0)\n    if not pos or not neg:\n        return np.nan\n    return float((rankdata(p)[y == 1].sum() - pos * (pos + 1) / 2) / (pos * neg))\n\n\ndef aucs(y, p):\n    return np.array([auc(y[:, j], p[:, j]) for j in range(len(TARGETS))])\n\n\ndef prediction_delta(baseline, candidate):\n    baseline = align(baseline)\n    candidate = align(candidate, baseline[UID])\n    b, c = (x[TARGETS].to_numpy(float) for x in (baseline, candidate))\n    rb, rc = (rankdata(x, axis=0) for x in (b, c))\n    return {\n        \'studies\': len(b), \'values_exactly_equal\': bool(np.array_equal(b, c)),\n        \'all_target_rankings_equal\': bool(np.array_equal(rb, rc)),\n        \'targets\': {t: {\n            \'max_absolute_change\': float(np.max(np.abs(c[:, j] - b[:, j]))),\n            \'mean_absolute_change\': float(np.mean(np.abs(c[:, j] - b[:, j]))),\n            \'rank_changed_rows\': int(np.sum(rb[:, j] != rc[:, j])),\n            \'baseline_unique_values\': int(len(np.unique(b[:, j]))),\n            \'candidate_unique_values\': int(len(np.unique(c[:, j]))),\n        } for j, t in enumerate(TARGETS)},\n        \'interpretation\': \'Identical ranks imply identical AUC on any fixed labels. Changed ranks do not imply improvement.\',\n    }\n\n\ndef require_changed_rankings(baseline, candidate):\n    """A candidate must change same-study rankings; this is NOT an accuracy gate."""\n    result = prediction_delta(baseline, candidate)\n    if result[\'values_exactly_equal\']:\n        raise ValueError(\'Candidate is numerically identical to the control\')\n    if result[\'all_target_rankings_equal\']:\n        raise ValueError(\'Candidate values differ but all rankings are unchanged; AUC cannot improve\')\n    return result\n\n\ndef compare(labels, baseline, candidate, bootstrap=2000, seed=1400, groups=None):\n    labels = align(labels, labels=True)\n    baseline, candidate = (align(x, labels[UID]) for x in (baseline, candidate))\n    y, b, c = (x[TARGETS].to_numpy(float) for x in (labels, baseline, candidate))\n    ba, ca = aucs(y, b), aucs(y, c)\n    if not np.isfinite(ba).all():\n        raise ValueError(\'All 12 targets need positive and negative labels; no silent macro target dropping\')\n    if bootstrap < 100:\n        raise ValueError(\'Use at least 100 paired bootstrap replicates\')\n    if groups is None:\n        clusters = [np.array([i]) for i in range(len(y))]\n    else:\n        if len(groups) != len(y) or pd.isna(groups).any():\n            raise ValueError(\'Group IDs must be aligned and nonmissing\')\n        codes, _ = pd.factorize(groups)\n        clusters = [np.flatnonzero(codes == k) for k in np.unique(codes)]\n    if len(clusters) < 20:\n        raise ValueError(\'Fewer than 20 independent resampling units; collect more confirmation data\')\n    rng = np.random.default_rng(seed)\n    draws = []\n    for _ in range(bootstrap):\n        idx = np.concatenate([clusters[k] for k in rng.integers(0, len(clusters), len(clusters))])\n        d = aucs(y[idx], c[idx]) - aucs(y[idx], b[idx])\n        if np.isfinite(d).all():\n            draws.append(d)\n    if len(draws) < .8 * bootstrap:\n        raise ValueError(\'Too many resamples lack both classes; confirmation cohort is insufficient\')\n    draws = np.asarray(draws)\n    ci = np.quantile(draws.mean(axis=1), [.025, .975])\n    target_ci = np.quantile(draws, [.025, .975], axis=0)\n    return {\n        \'scope\': \'Supplied cohort only. Not hidden-test performance; not proof of training exclusion.\',\n        \'baseline_macro_auc\': float(ba.mean()), \'candidate_macro_auc\': float(ca.mean()),\n        \'macro_delta\': float((ca - ba).mean()), \'macro_delta_ci95\': ci.tolist(),\n        \'point_gain_at_least_0_02\': bool((ca - ba).mean() >= .02),\n        \'ci_lower_bound_at_least_0_02\': bool(ci[0] >= .02),\n        \'bootstrap_unit\': \'group\' if groups is not None else \'study\',\n        \'independent_units\': len(clusters), \'valid_replicates\': len(draws),\n        \'requested_replicates\': bootstrap, \'seed\': seed,\n        \'ci_caveat\': \'Paired percentile intervals; degenerate replicates excluded; not corrected for candidate selection.\',\n        \'targets\': {t: {\'baseline_auc\': float(ba[j]), \'candidate_auc\': float(ca[j]),\n            \'delta\': float(ca[j] - ba[j]), \'delta_ci95\': target_ci[:, j].tolist(),\n            \'positive\': int(np.sum(y[:, j] == 1)), \'negative\': int(np.sum(y[:, j] == 0)),\n            \'missing\': int(np.isnan(y[:, j]).sum()),\n            \'perfect_target_max_macro_gain\': float((1 - ba[j]) / len(TARGETS))}\n            for j, t in enumerate(TARGETS)},\n        \'prediction_changes\': prediction_delta(baseline, candidate),\n    }\n\n\ndef write_new(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open(\'x\', encoding=\'utf-8\') as stream:\n        json.dump(data, stream, indent=2, allow_nan=False)\n        stream.write(\'\\n\')\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    for name in (\'baseline\', \'candidate\', \'output\'):\n        p.add_argument(\'--\' + name, type=Path, required=True)\n    p.add_argument(\'--labels\', type=Path)\n    p.add_argument(\'--groups\', type=Path, help=\'CSV with StudyInstanceUID,GroupID\')\n    p.add_argument(\'--bootstrap\', type=int, default=2000)\n    args = p.parse_args()\n    frames = [pd.read_csv(x, dtype={UID: str}) for x in (args.baseline, args.candidate)]\n    result = prediction_delta(*frames)\n    if args.labels:\n        labels = pd.read_csv(args.labels, dtype={UID: str})\n        groups = None\n        if args.groups:\n            g = pd.read_csv(args.groups, dtype=str)\n            if g[UID].duplicated().any() or set(g[UID]) != set(labels[UID]):\n                raise ValueError(\'Group table coverage mismatch\')\n            groups = g.set_index(UID).loc[labels[UID], \'GroupID\'].to_numpy()\n        result = compare(labels, *frames, bootstrap=args.bootstrap, groups=groups)\n    elif args.groups:\n        raise ValueError(\'--groups requires --labels\')\n    result[\'sha256\'] = {k: digest(v) for k in (\'baseline\', \'candidate\', \'labels\', \'groups\')\n                        if (v := getattr(args, k)) is not None}\n    write_new(args.output, result)\n    print(json.dumps({k: v for k, v in result.items() if k != \'targets\'}, indent=2))\n\n\nif __name__ == \'__main__\':\n    main()\n', diagnostics.__dict__)
clean_specialist = types.ModuleType('clean_specialist')
sys.modules['clean_specialist'] = clean_specialist
exec('"""Clean-reference nested grouped CV on frozen image features.\n\nNo V13 outputs/weights are inputs. Protocol is fixed before this pilot runs.\nPatientID grouping is provisional, and the historical gold set is development.\n"""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.special import expit\nfrom sklearn.metrics import roc_auc_score\n\nTARGETS = [\'ACL\', \'MCL\', \'Medial Meniscus\', \'Lateral Meniscus\', \'Medial OA\',\n           \'Lateral OA\', \'PF OA\', \'Effusion\', \'Synovitis\', "Baker\'s", \'Contusion\', \'Fracture\']\nFOCUS = [3, 5, 6, 8]\nPROTOCOL = {\n    \'name\': \'clean-dinov2-small-regional-pilot-v1\',\n    \'encoder\': \'facebook/dinov2-small\',\n    \'encoder_revision\': \'ed25f3a31f01632728cabb09d1542f84ab7b0056\',\n    \'encoder_frozen\': True, \'expert_labels_only\': True,\n    \'outer_folds\': 5, \'inner_folds\': 3, \'seed\': 1401,\n    \'regularization_grid\': [1.0, 0.1, 0.01],\n    \'blend_grid\': [0.0, 0.25, 0.5, 1.0],\n    \'inner_auc_min_gain\': 0.01, \'inner_bce_max_regression\': 0.01,\n    \'fixed_blend_weight\': 0.25, \'focus_targets\': [TARGETS[j] for j in FOCUS],\n    \'image_size\': 336, \'slices_per_series\': 12, \'max_series_per_plane\': 2,\n    \'slice_quantiles\': [0.1, 0.9], \'planes\': [\'Sagittal\', \'Coronal\', \'Axial\'],\n    \'regional_features\': \'All four image-coordinate patch quadrants, per plane; not anatomical segmentation\',\n    \'primary\': \'inner_selected_specialist\', \'secondary\': [\'fixed_25pct_specialist\', \'regional_only_focus\'],\n    \'patient_identity_semantics_verified\': False, \'exact_v13_comparator\': False,\n    \'untouched_confirmation\': False, \'auto_promotion\': False,\n}\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef splits(groups, folds, seed):\n    groups = np.asarray(groups, str)\n    if groups.ndim != 1 or any(not s.strip() or s.lower() in {\'nan\', \'none\'} for s in groups):\n        raise ValueError(\'Nonmissing group IDs required\')\n    unique = sorted(set(groups), key=lambda g: hashlib.sha256(f\'{seed}|{g}\'.encode()).hexdigest())\n    if folds < 2 or len(unique) < folds:\n        raise ValueError(\'Insufficient distinct groups\')\n    # Label-independent balanced group allocation; repeated examinations stay together.\n    buckets, counts = {}, np.zeros(folds, int)\n    for g in unique:\n        k = int(np.argmin(counts))\n        buckets[g] = k\n        counts[k] += int(np.sum(groups == g))\n    assignment = np.array([buckets[g] for g in groups])\n    return [(np.flatnonzero(assignment != k), np.flatnonzero(assignment == k)) for k in range(folds)]\n\n\ndef bce(y, pred):\n    p = np.clip(pred, 1e-7, 1 - 1e-7)\n    return float(np.mean(-y * np.log(p) - (1-y) * np.log1p(-p)))\n\n\ndef fit_head(x, y, regularization):\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    if x.ndim != 2 or y.ndim != 2 or len(x) != len(y) or not len(x) or not x.shape[1]:\n        raise ValueError(\'Invalid feature/label shapes\')\n    if not np.isfinite(x).all() or not np.isin(y, [0, 1]).all():\n        raise ValueError(\'Finite features and expert binary labels required\')\n    lam = np.broadcast_to(np.asarray(regularization, float), (y.shape[1],))\n    if not np.isfinite(lam).all() or (lam <= 0).any():\n        raise ValueError(\'Positive finite regularization required\')\n    mean, scale = x.mean(0), x.std(0)\n    scale = np.where(scale > 1e-6, scale, 1.)\n    z = (x - mean) / scale / np.sqrt(x.shape[1])\n    # Exact train-row-space representation, NOT truncated PCA. Any optimal\n    # L2-regularized linear head lies in this span. Scaling is training-only.\n    eig, u = np.linalg.eigh(z @ z.T)\n    keep = eig > max(float(eig.max()), 1.) * 1e-10\n    transform = z.T @ (u[:, keep] / np.sqrt(eig[keep]))\n    reduced = z @ transform\n    weight, intercept = np.zeros((x.shape[1], y.shape[1])), np.zeros(y.shape[1])\n    supported = np.zeros(y.shape[1], bool)\n    for j in range(y.shape[1]):\n        target = y[:, j]\n        prior = (target.sum() + 1) / (len(target) + 2)\n        intercept[j] = np.log(prior / (1-prior))\n        if min(target.sum(), len(target)-target.sum()) < 2:\n            continue\n        def objective(theta):\n            logits = reduced @ theta[:-1] + theta[-1]\n            error = (expit(logits) - target) / len(target)\n            loss = np.mean(np.logaddexp(0, logits) - target * logits) + lam[j] * (theta[:-1] @ theta[:-1]) / 2\n            grad = np.r_[reduced.T @ error + lam[j] * theta[:-1], error.sum()]\n            return loss, grad\n        initial = np.r_[np.zeros(reduced.shape[1]), intercept[j]]\n        result = minimize(objective, initial, method=\'L-BFGS-B\', jac=True,\n                          options={\'maxiter\': 300, \'ftol\': 1e-11, \'gtol\': 1e-7})\n        if not result.success:\n            raise RuntimeError(f\'Head optimization failed: {result.message}\')\n        weight[:, j], intercept[j], supported[j] = transform @ result.x[:-1], result.x[-1], True\n    return {\'mean\': mean, \'scale\': scale, \'weight\': weight, \'intercept\': intercept,\n            \'supported\': supported, \'regularization\': lam.copy()}\n\n\ndef predict_head(model, x):\n    x = np.asarray(x, float)\n    if x.ndim != 2 or x.shape[1] != len(model[\'mean\']) or not np.isfinite(x).all():\n        raise ValueError(\'Feature contract mismatch\')\n    return expit(((x-model[\'mean\']) / model[\'scale\'] / np.sqrt(x.shape[1])) @ model[\'weight\'] + model[\'intercept\'])\n\n\ndef tune_heads(x, y, groups, seed, protocol=PROTOCOL):\n    grid = protocol[\'regularization_grid\']\n    pred = np.empty((len(grid), len(y), y.shape[1]))\n    fold_log = []\n    for k, (tr, va) in enumerate(splits(groups, protocol[\'inner_folds\'], seed)):\n        assert not set(groups[tr]) & set(groups[va])\n        fold_log.append({\'fold\': k, \'train_groups\': sorted(set(groups[tr])), \'valid_groups\': sorted(set(groups[va]))})\n        for i, lam in enumerate(grid):\n            model = fit_head(x[tr], y[tr], lam)\n            pred[i, va] = predict_head(model, x[va])\n    # AUC for model ranking, log loss only for tie break; ties favor stronger L2.\n    choices, selected, scores = [], np.empty_like(y, dtype=float), []\n    for j in range(y.shape[1]):\n        values = []\n        for i, lam in enumerate(grid):\n            auc = float(roc_auc_score(y[:, j], pred[i, :, j])) if len(np.unique(y[:, j])) == 2 else 0.5\n            values.append({\'regularization\': lam, \'auc\': auc, \'bce\': bce(y[:, j], pred[i, :, j])})\n        best = max(range(len(grid)), key=lambda i: (values[i][\'auc\'], -values[i][\'bce\'], grid[i]))\n        choices.append(grid[best])\n        selected[:, j] = pred[best, :, j]\n        scores.append(values)\n    return np.array(choices), selected, {\'inner_folds\': fold_log, \'grid_scores\': scores}\n\n\ndef train_outer(xbase, xregion, y, groups, train, valid, seed, protocol=PROTOCOL):\n    # No y[valid] access in this function: even alpha and scale fitting stay inside train.\n    if set(groups[train]) & set(groups[valid]):\n        raise ValueError(\'Group overlap\')\n    base_lam, inner_base, base_log = tune_heads(xbase[train], y[train], groups[train], seed, protocol)\n    reg_lam, inner_reg, reg_log = tune_heads(xregion[train], y[train][:, FOCUS], groups[train], seed, protocol)\n    base_model = fit_head(xbase[train], y[train], base_lam)\n    reg_model = fit_head(xregion[train], y[train][:, FOCUS], reg_lam)\n    base, regional = predict_head(base_model, xbase[valid]), predict_head(reg_model, xregion[valid])\n    selected, fixed, raw = base.copy(), base.copy(), base.copy()\n    alpha, decisions = [], []\n    for jj, j in enumerate(FOCUS):\n        t, b, r = y[train, j], inner_base[:, j], inner_reg[:, jj]\n        base_auc = float(roc_auc_score(t, b)) if len(np.unique(t)) == 2 else 0.5\n        candidates = []\n        for a in protocol[\'blend_grid\']:\n            mixed = (1-a)*b + a*r\n            score = float(roc_auc_score(t, mixed)) if len(np.unique(t)) == 2 else 0.5\n            eligible = a == 0 or (score-base_auc >= protocol[\'inner_auc_min_gain\'] and\n                                  bce(t, mixed)-bce(t, b) <= protocol[\'inner_bce_max_regression\'])\n            candidates.append({\'alpha\': a, \'auc\': score, \'bce\': bce(t, mixed), \'eligible\': eligible})\n        choice = max((c for c in candidates if c[\'eligible\']), key=lambda c: (c[\'auc\'], -c[\'alpha\']))[\'alpha\']\n        # Unsupported regional heads cannot replace a supported baseline.\n        if not reg_model[\'supported\'][jj]:\n            choice = 0.\n        alpha.append(choice)\n        selected[:, j] = (1-choice)*base[:, j] + choice*regional[:, jj]\n        fixed[:, j] = (1-protocol[\'fixed_blend_weight\'])*base[:, j] + protocol[\'fixed_blend_weight\']*regional[:, jj]\n        raw[:, j] = regional[:, jj]\n        decisions.append({\'target\': TARGETS[j], \'selected_alpha\': choice, \'candidates\': candidates})\n    log = {\'train_groups\': sorted(set(groups[train])), \'valid_groups\': sorted(set(groups[valid])),\n           \'base_regularization\': base_lam.tolist(), \'regional_regularization\': reg_lam.tolist(),\n           \'base_inner\': base_log, \'regional_inner\': reg_log, \'blend_decisions\': decisions}\n    return {\'reference\': base, \'inner_selected_specialist\': selected,\n            \'fixed_25pct_specialist\': fixed, \'regional_only_focus\': raw}, base_model, reg_model, np.array(alpha), log\n\n\ndef evaluate_features(features, output, protocol=PROTOCOL):\n    from diagnostics import compare\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    with np.load(features, allow_pickle=False) as data:\n        ids, groups = data[\'StudyInstanceUID\'].astype(str), data[\'GroupID\'].astype(str)\n        xb, xr, y = data[\'base\'].astype(float), data[\'regional\'].astype(float), data[\'labels\'].astype(float)\n    if len(set(ids)) != len(ids) or len(ids) != len(groups) or y.shape != (len(ids), 12):\n        raise ValueError(\'Invalid feature archive alignment\')\n    if len(xb) != len(y) or len(xr) != len(y) or not np.isin(y, [0, 1]).all():\n        raise ValueError(\'Feature/label alignment or expert label contract failed\')\n    names = [\'reference\', \'inner_selected_specialist\', \'fixed_25pct_specialist\', \'regional_only_focus\']\n    oof = {name: np.full_like(y, np.nan) for name in names}\n    assignment, logs = np.full(len(y), -1), []\n    for k, (tr, va) in enumerate(splits(groups, protocol[\'outer_folds\'], protocol[\'seed\'])):\n        pred, bm, rm, alpha, log = train_outer(xb, xr, y, groups, tr, va, protocol[\'seed\']+k+1, protocol)\n        for name in names:\n            oof[name][va] = pred[name]\n        assignment[va] = k\n        np.savez_compressed(output/f\'fold_{k}_reference.npz\', **bm)\n        np.savez_compressed(output/f\'fold_{k}_regional.npz\', **rm, alpha=alpha, target_indices=np.array(FOCUS))\n        log.update(fold=k, train_ids=ids[tr].tolist(), valid_ids=ids[va].tolist())\n        logs.append(log)\n        print(f\'CLEAN HEAD TRAINING outer fold {k+1}/{protocol["outer_folds"]} complete; train={len(tr)} valid={len(va)} alpha={alpha.tolist()}\', flush=True)\n    if any(not np.isfinite(p).all() for p in oof.values()) or (assignment < 0).any():\n        raise ValueError(\'Incomplete OOF predictions\')\n    def frame(values):\n        f = pd.DataFrame(values, columns=TARGETS)\n        f.insert(0, \'StudyInstanceUID\', ids)\n        return f\n    label_frame = frame(y)\n    label_frame.to_csv(output/\'gold_labels.csv\', index=False)\n    pd.DataFrame({\'StudyInstanceUID\': ids, \'GroupID\': groups, \'fold\': assignment}).to_csv(output/\'folds.csv\', index=False)\n    for name, pred in oof.items():\n        frame(pred).to_csv(output/f\'{name}_oof.csv\', index=False)\n    results = {}\n    for name in names[1:]:\n        report = compare(label_frame, frame(oof[\'reference\']), frame(oof[name]), groups=groups, bootstrap=2000, seed=1401)\n        report[\'calibration\'] = {kind: {\'bce\': bce(y, oof[n]), \'brier\': float(np.mean((oof[n]-y)**2))}\n                                 for kind, n in [(\'reference\', \'reference\'), (\'candidate\', name)]}\n        report[\'ci_caveat\'] += \' Fixed OOF prediction resampling does not refit models; training/fold uncertainty is omitted. Historical gold and architecture selection make this exploratory.\'\n        report[\'unchanged_other_eight_targets\'] = bool(np.array_equal(oof[name][:, [j for j in range(12) if j not in FOCUS]], oof[\'reference\'][:, [j for j in range(12) if j not in FOCUS]]))\n        report[\'changed_values\'] = int(np.count_nonzero(oof[name] != oof[\'reference\']))\n        fold_auc = []\n        for k in range(protocol[\'outer_folds\']):\n            idx = assignment == k\n            valid_targets = [j for j in range(12) if len(np.unique(y[idx, j])) == 2]\n            fold_auc.append({\'fold\': k, \'studies\': int(idx.sum()), \'scorable_targets\': [TARGETS[j] for j in valid_targets],\n                             \'target_deltas\': {TARGETS[j]: float(roc_auc_score(y[idx, j], oof[name][idx, j])-roc_auc_score(y[idx, j], oof[\'reference\'][idx, j])) for j in valid_targets},\n                             \'not_a_complete_macro_auc\': len(valid_targets) != 12})\n        report[\'within_fold_diagnostics\'] = fold_auc\n        results[name] = report\n    receipt = {\'status\': \'COMPLETED_PROVISIONAL_GROUPED_RESEARCH\', \'protocol\': protocol,\n               \'studies\': len(ids), \'groups\': len(set(groups)), \'new_head_training_performed\': True,\n               \'saved_reference_heads\': protocol[\'outer_folds\'], \'saved_regional_heads\': protocol[\'outer_folds\'],\n               \'outer_label_isolation\': \'Outer validation labels were not passed to fitting/selection; feature encoder fixed and frozen\',\n               \'feature_sha256\': sha(features), \'folds\': logs, \'results\': results,\n               \'patient_independence_confirmed\': False, \'v13_improvement_verified\': False,\n               \'leaderboard_prediction\': None, \'leaderboard_submission\': False,\n               \'limitations\': [\'58 historical expert-labeled studies only; no weak report labels or V13 models used\',\n                               \'PatientID tags are anonymized singletons; repeat-patient linkage unverified\',\n                               \'Source encoder is public external pretraining; exact image-level membership is not independently audited\',\n                               \'Pooled OOF AUC may include cross-fold calibration differences; within-fold diagnostics also provided\',\n                               \'The three fixed candidate definitions must all be reported; no post-hoc winner deployment\']}\n    receipt[\'artifact_sha256\'] = {p.name: sha(p) for p in output.iterdir() if p.is_file()}\n    (output/\'training_results.json\').write_text(json.dumps(receipt, indent=2, allow_nan=False)+\'\\n\', encoding=\'utf-8\')\n    print(json.dumps({name: {k: r[k] for k in [\'baseline_macro_auc\', \'candidate_macro_auc\', \'macro_delta\', \'macro_delta_ci95\', \'changed_values\']} for name, r in results.items()}, indent=2), flush=True)\n    return receipt\n', clean_specialist.__dict__)
clean_features = types.ModuleType('clean_features')
sys.modules['clean_features'] = clean_features
exec('"""Frozen DINOv2 features from real MRI DICOMs; no report or model-label fitting."""\nimport hashlib\nimport json\nimport time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\nfrom clean_specialist import PROTOCOL, TARGETS, sha\n\n\ndef pixels_to_rgb(ds, size=336):\n    a = ds.pixel_array.astype(np.float32)\n    if a.ndim != 2 or min(a.shape) < 16 or not np.isfinite(a).all():\n        raise ValueError(\'Expected finite single-frame grayscale MRI\')\n    a = a * float(ds.get(\'RescaleSlope\', 1)) + float(ds.get(\'RescaleIntercept\', 0))\n    lo, hi = np.percentile(a, [1, 99])\n    if hi <= lo:\n        raise ValueError(\'Constant/degenerate slice\')\n    a = np.clip((a-lo)/(hi-lo), 0, 1)\n    if str(ds.get(\'PhotometricInterpretation\', \'MONOCHROME2\')) == \'MONOCHROME1\':\n        a = 1-a\n    # Preserve the entire FOV/aspect ratio. No learned/case-label-dependent crop.\n    im = Image.fromarray(np.uint8(a*255)).convert(\'RGB\')\n    ratio = size / max(im.size)\n    im = im.resize((max(1, round(im.width*ratio)), max(1, round(im.height*ratio))), Image.Resampling.BILINEAR)\n    out = Image.new(\'RGB\', (size, size))\n    out.paste(im, ((size-im.width)//2, (size-im.height)//2))\n    return np.asarray(out).transpose(2, 0, 1).astype(np.float32)/255\n\n\ndef ordered_files(folder, uid, sid):\n    import pydicom\n    records = []\n    for file in sorted(folder.glob(\'*.dcm\')):\n        ds = pydicom.dcmread(file, stop_before_pixels=True,\n                            specific_tags=[\'StudyInstanceUID\', \'SeriesInstanceUID\', \'ImagePositionPatient\',\n                                           \'ImageOrientationPatient\', \'InstanceNumber\'])\n        if str(ds.get(\'StudyInstanceUID\', \'\')) != uid or str(ds.get(\'SeriesInstanceUID\', \'\')) != sid:\n            raise ValueError(\'DICOM/header path UID mismatch\')\n        orientation = np.asarray(ds.get(\'ImageOrientationPatient\', []), float)\n        position = np.asarray(ds.get(\'ImagePositionPatient\', []), float)\n        records.append((file, orientation, position, float(ds.get(\'InstanceNumber\', len(records)))))\n    if not records:\n        raise ValueError(\'No DICOMs in selected series\')\n    physical = all(o.shape == (6,) and p.shape == (3,) and np.isfinite(o).all() and np.isfinite(p).all()\n                   for _, o, p, _ in records)\n    if physical:\n        reference = records[0][1]\n        physical = all(np.allclose(o, reference, atol=.01) for _, o, _, _ in records)\n    if physical:\n        normal = np.cross(reference[:3], reference[3:])\n        physical = np.linalg.norm(normal) > .9\n    records.sort(key=lambda r: (float(r[2] @ normal) if physical else r[3], r[0].name))\n    return [r[0] for r in records], \'physical_position\' if physical else \'instance_number_fallback\'\n\n\ndef extract(root, groups, output, protocol=PROTOCOL):\n    import pydicom\n    import torch\n    from transformers import AutoModel\n    from huggingface_hub import snapshot_download\n    start = time.perf_counter()\n    root, output = Path(root), Path(output)\n    output.mkdir(parents=True, exist_ok=False)\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'GPU required for bounded extraction; do not silently run hours on CPU\')\n    torch.manual_seed(protocol[\'seed\'])\n    torch.set_num_threads(2)\n    train = pd.read_csv(root/\'train.csv\', dtype={\'StudyInstanceUID\': str})\n    series = pd.read_csv(root/\'train_series.csv\', dtype={\'StudyInstanceUID\': str, \'SeriesInstanceUID\': str})\n    expected_hash = {\'train.csv\': \'8ca2203c0e9d61c080c7a314c7cdb51c1b03a1d9eb4770819f7f34af53ef4e33\',\n                     \'train_series.csv\': \'573c1d80772bf41211c91b149c95677385a1c22d63f485c347f1b46c0177aef3\'}\n    if any(sha(root/name) != digest for name, digest in expected_hash.items()):\n        raise ValueError(\'Competition tables differ from audited dataset version\')\n    gold = train.loc[train[TARGETS].notna().all(axis=1), [\'StudyInstanceUID\']+TARGETS].sort_values(\'StudyInstanceUID\')\n    if len(gold) != 58 or set(gold.StudyInstanceUID) != set(groups):\n        raise ValueError(\'Expected exact previously audited 58 expert studies\')\n    if not np.isin(gold[TARGETS].to_numpy(), [0, 1]).all():\n        raise ValueError(\'Gold label schema changed\')\n    required = [\'Anatomical_Plane\', \'Fluid_Sensitive\', \'Fat_Suppression\']\n    if not set(required).issubset(series):\n        raise ValueError(\'Series metadata schema changed\')\n    snapshot = Path(snapshot_download(protocol[\'encoder\'], revision=protocol[\'encoder_revision\'],\n                                     allow_patterns=[\'config.json\', \'model.safetensors\'], token=False))\n    model = AutoModel.from_pretrained(snapshot, local_files_only=True, use_safetensors=True, trust_remote_code=False)\n    if model.config.model_type != \'dinov2\' or model.config.hidden_size != 384 or model.config.patch_size != 14:\n        raise ValueError(\'Unexpected pretrained architecture\')\n    model.requires_grad_(False).eval().cuda()\n    weight_hash = sha(snapshot/\'model.safetensors\')\n    mean = torch.tensor([.485, .456, .406], device=\'cuda\')[None, :, None, None]\n    std = torch.tensor([.229, .224, .225], device=\'cuda\')[None, :, None, None]\n    all_base, all_region, audit, pixel_owners = [], [], [], {}\n    size, hidden = protocol[\'image_size\'], model.config.hidden_size\n    grid = size//14\n    def read_pixels(item):\n        file, uid, sid = item\n        ds = pydicom.dcmread(file)\n        if str(ds.get(\'StudyInstanceUID\', \'\')) != uid or str(ds.get(\'SeriesInstanceUID\', \'\')) != sid:\n            raise ValueError(\'Pixel DICOM UID mismatch\')\n        # Encoded pixel payload hash catches exact duplicate image payloads across groups.\n        pixel_hash = hashlib.sha256(ds.PixelData).hexdigest()\n        return pixels_to_rgb(ds, size), pixel_hash\n    with ThreadPoolExecutor(max_workers=6) as pool:\n        for ii, uid in enumerate(gold.StudyInstanceUID):\n            s = series[series.StudyInstanceUID == uid].copy()\n            base_parts, regional_parts, details = [], [], []\n            for plane in protocol[\'planes\']:\n                rows = s[s.Anatomical_Plane.str.casefold() == plane.casefold()].copy()\n                rows[\'priority\'] = pd.to_numeric(rows.Fluid_Sensitive, errors=\'coerce\').fillna(0)*2 + pd.to_numeric(rows.Fat_Suppression, errors=\'coerce\').fillna(0)\n                rows = rows.sort_values([\'priority\', \'SeriesInstanceUID\'], ascending=[False, True]).head(protocol[\'max_series_per_plane\'])\n                cls_series, region_series = [], []\n                for sid in rows.SeriesInstanceUID:\n                    files, ordering = ordered_files(root/\'train_series\'/uid/sid, uid, sid)\n                    idx = np.unique(np.rint(np.linspace(protocol[\'slice_quantiles\'][0]*(len(files)-1),\n                                                         protocol[\'slice_quantiles\'][1]*(len(files)-1),\n                                                         min(protocol[\'slices_per_series\'], len(files)))).astype(int))\n                    batch = list(pool.map(read_pixels, [(files[i], uid, sid) for i in idx]))\n                    for _, pixel_hash in batch:\n                        if pixel_hash in pixel_owners and pixel_owners[pixel_hash] != groups[uid]:\n                            raise ValueError(\'Exact sampled image duplicate across PatientID groups; regroup before fitting\')\n                        pixel_owners[pixel_hash] = groups[uid]\n                    x = torch.from_numpy(np.stack([a for a, _ in batch])).cuda()\n                    with torch.inference_mode(), torch.autocast(\'cuda\', dtype=torch.float16):\n                        h = model(pixel_values=(x-mean)/std).last_hidden_state\n                    if h.shape != (len(batch), 1+grid*grid, hidden):\n                        raise ValueError(\'Unexpected patch-grid shape\')\n                    h = h.float().cpu().numpy()\n                    cls = h[:, 0]\n                    patches = h[:, 1:].reshape(len(batch), grid, grid, hidden)\n                    quadrants = np.stack([patches[:, ra:rb, ca:cb].mean((1,2))\n                                          for ra, rb in [(0, grid//2), (grid//2, grid)]\n                                          for ca, cb in [(0, grid//2), (grid//2, grid)]], axis=1)\n                    cls_series.append(np.concatenate([cls.mean(0), cls.max(0)]))\n                    region_series.append(np.concatenate([quadrants.mean(0).ravel(), quadrants.max(0).ravel()]))\n                    details.append({\'plane\': plane, \'SeriesInstanceUID\': sid, \'available_slices\': len(files),\n                                    \'used_slices\': len(idx), \'ordering\': ordering,\n                                    \'selected_instance_files\': [files[i].name for i in idx]})\n                present = float(bool(cls_series))\n                base_parts.append(np.r_[np.mean(cls_series, axis=0) if cls_series else np.zeros(hidden*2), present])\n                regional_parts.append(np.r_[np.mean(region_series, axis=0) if region_series else np.zeros(hidden*8), present])\n            if not details:\n                raise ValueError(\'Study has no usable canonical-plane series; do not silently drop it\')\n            all_base.append(np.concatenate(base_parts))\n            # Include global features in specialist input; no artificial information removal.\n            all_region.append(np.concatenate(base_parts+regional_parts))\n            audit.append({\'StudyInstanceUID\': uid, \'series\': details})\n            print(f\'CLEAN FEATURES {ii+1}/58 studies, elapsed={time.perf_counter()-start:.1f}s\', flush=True)\n    base, regional = np.asarray(all_base, np.float32), np.asarray(all_region, np.float32)\n    if not np.isfinite(base).all() or not np.isfinite(regional).all():\n        raise ValueError(\'Nonfinite features\')\n    feature_file = output/\'clean_features.npz\'\n    np.savez_compressed(feature_file, StudyInstanceUID=gold.StudyInstanceUID.to_numpy(str),\n                        GroupID=np.array([groups[u] for u in gold.StudyInstanceUID]),\n                        base=base, regional=regional, labels=gold[TARGETS].to_numpy(np.float32))\n    receipt = {\'protocol\': protocol, \'encoder_weights_sha256\': weight_hash,\n               \'encoder_config_sha256\': sha(snapshot/\'config.json\'), \'encoder_snapshot_revision\': snapshot.name,\n               \'encoder_training_on_rsna_in_this_experiment\': False, \'parameters_require_grad\': sum(p.numel() for p in model.parameters() if p.requires_grad),\n               \'input_sha256\': expected_hash, \'feature_sha256\': sha(feature_file),\n               \'base_shape\': list(base.shape), \'regional_shape\': list(regional.shape),\n               \'sampled_images\': sum(d[\'used_slices\'] for a in audit for d in a[\'series\']),\n               \'sampled_exact_pixel_duplicate_cross_group_check\': \'passed\',\n               \'scope_of_duplicate_check\': \'Encoded pixel payloads in sampled slices only; not a near-duplicate or full-volume audit\',\n               \'elapsed_seconds\': time.perf_counter()-start, \'gpu\': torch.cuda.get_device_name(0),\n               \'torch_version\': torch.__version__, \'series_audit\': audit}\n    (output/\'feature_receipt.json\').write_text(json.dumps(receipt, indent=2)+\'\\n\', encoding=\'utf-8\')\n    return feature_file\n', clean_features.__dict__)


In [ ]:
SOURCE_HASHES = {'diagnostics': 'eb64dbb888411742a3e9c42393d3c4588e6e4bea4ef98f044dca02c6643ae0bb', 'clean_specialist': '639028f1f435f28589561e1265f784b4e4f372024271727932a62c5e837ecea1', 'clean_features': '0433e07294242178dbdce3474d37c85b8e066ea2b92d393a8b9e252795c96193'}
from pathlib import Path
import json, time, zipfile
import pandas as pd
import numpy as np
from threadpoolctl import threadpool_limits
from clean_specialist import PROTOCOL, TARGETS, sha, splits
start = time.perf_counter()
output = Path('/kaggle/working/clean_specialist')
output.mkdir(exist_ok=False)
root_candidates = [Path('/kaggle/input/rsna-knee-abnormality-detection'),
                   Path('/kaggle/input/competitions/rsna-knee-abnormality-detection')]
roots = [p for p in root_candidates if (p/'train.csv').exists() and (p/'train_series.csv').exists()]
if len(roots) != 1:
    raise RuntimeError('Attach only the competition plus patient-audit notebook output')
map_candidates = list(Path('/kaggle/input').glob('*/patient_audit/patient_tag_groups.csv'))
map_candidates += list(Path('/kaggle/input').glob('notebooks/*/*/patient_audit/patient_tag_groups.csv'))
maps = [p for p in map_candidates
        if sha(p) == '83a1c3945988d83241319678dc815828e3d1fbb2b33b1276c2b3d38ecf091df4']
if len(maps) != 1:
    raise RuntimeError('Attach RSNA Knee V14 Patient Audit version 346132000 output')
mapping = pd.read_csv(maps[0], dtype=str)
if mapping.StudyInstanceUID.duplicated().any() or mapping.isna().any().any():
    raise ValueError('Invalid PatientID grouping map')
train = pd.read_csv(roots[0]/'train.csv', dtype={'StudyInstanceUID': str})
ids = train.loc[train[TARGETS].notna().all(axis=1), 'StudyInstanceUID'].sort_values().to_numpy(str)
groups = mapping.set_index('StudyInstanceUID').loc[ids, 'GroupID'].to_dict()
# Freeze this recipe and partitions before model download, image extraction or fits.
assignment = np.full(len(ids), -1)
for k, (_, va) in enumerate(splits(np.array([groups[u] for u in ids]), PROTOCOL['outer_folds'], PROTOCOL['seed'])):
    assignment[va] = k
pd.DataFrame({'StudyInstanceUID': ids, 'GroupID': [groups[u] for u in ids], 'fold': assignment}).to_csv(output/'predeclared_folds.csv', index=False)
(output/'predeclared_protocol.json').write_text(json.dumps(PROTOCOL, indent=2)+'\n')
print('PROVISIONAL CLEAN REFERENCE: no V13 weights, no weak/report labels, no leaderboard submission', flush=True)
features = clean_features.extract(roots[0], groups, output/'features')
with threadpool_limits(limits=2):
    results = clean_specialist.evaluate_features(features, output/'evaluation')
runtime = {'wall_seconds': time.perf_counter()-start, 'new_training_completed': True,
           'patient_independence_confirmed': False, 'v13_gain_verified': False,
           'source_sha256': SOURCE_HASHES, 'recipe_sha256': sha(output/'predeclared_protocol.json')}
(output/'run_receipt.json').write_text(json.dumps(runtime, indent=2)+'\n')
with zipfile.ZipFile('/kaggle/working/clean_specialist_artifacts.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for file in sorted(output.rglob('*')):
        if file.is_file():
            archive.write(file, file.relative_to(output))
print('CLEAN SPECIALIST TRAINING AND BENCHMARK COMPLETE', json.dumps(runtime), flush=True)
